In [2]:
pip install pymupdf


DEPRECATION: Loading egg at /Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/jupyter-1.0.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
Note: you may need to restart the kernel to use updated packages.


In [3]:
import fitz
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
from nltk.corpus import wordnet
import string
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd
from gensim.models import Word2Vec, FastText

In [49]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


[nltk_data] Downloading package punkt to /Users/pranesh_s/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/pranesh_s/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/pranesh_s/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/pranesh_s/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

### Preprocess for Skipgram and CBOW

In [7]:
def preprocess_text(text):
    text = text.lower()
#     print("After Lowercasing:", text)
#     print("\n\n\n\n")
    tokens = word_tokenize(text)
#     print("After Tokenization:", tokens)
#     print("\n\n\n\n")
    stop_words = set(stopwords.words('english'))
    tokens = [token for token in tokens if token not in stop_words]
#     print("After Stopword Removal:", tokens)
#     print("\n\n\n\n")
    tokens = [token for token in tokens if token not in string.punctuation]
#     print("After Punctuation Removal:", tokens)
#     print("\n\n\n\n")
    tokens = [token for token in tokens if not (token.isdigit() or (token[:-1].isdigit() and token[-1] == '.'))]
#     print("After Number Removal:", tokens)
#     print("\n\n\n\n")
    lemmatizer = WordNetLemmatizer()
    pos_tags = pos_tag(tokens)

    def get_wordnet_pos(treebank_tag):
        if treebank_tag.startswith('J'):
            return wordnet.ADJ
        elif treebank_tag.startswith('V'):
            return wordnet.VERB
        elif treebank_tag.startswith('N'):
            return wordnet.NOUN
        elif treebank_tag.startswith('R'):
            return wordnet.ADV
        else:
            return wordnet.NOUN

    lemmatized_tokens = [lemmatizer.lemmatize(token, get_wordnet_pos(pos_tag)) for token, pos_tag in pos_tags]
#     print("After Lemmatization:", lemmatized_tokens)

    lemmatized_tokens = [token for token in lemmatized_tokens if token] 
    
    return lemmatized_tokens

###Preprocess for Bag of words and Tf-Idf

In [ ]:
def preprocess_alt_text(text):

    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    tokens = []

    for sentence in text:
        if sentence:
            s = ''
            for word, pos_tag in nltk.pos_tag(word_tokenize(sentence)):
                if not any(char.isdigit() for char in word) and word.lower() not in stop_words:
                    word_without_punct = ''.join(char for char in word if char not in string.punctuation)
                    pos_tag = get_wordnet_pos(pos_tag)
                    lemma = lemmatizer.lemmatize(word_without_punct, pos=pos_tag)
                    s += lemma.lower() + ' '
            tokens.append(s.strip())  
        
    return tokens

def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return 'a'  
    elif treebank_tag.startswith('V'):
        return 'v'  
    elif treebank_tag.startswith('N'):
        return 'n' 
    elif treebank_tag.startswith('R'):
        return 'r'  
    else:
        return 'n'

###PDF-Text Extraction

In [9]:
def process_pdf(pdf_path):
    pdf_document = fitz.open(pdf_path)
    preprocessed_text_corpus = []
    altt_corpus=[]
    alt_corpus=[]
    for page_number in range(pdf_document.page_count):
        page = pdf_document[page_number]
        page_text = page.get_text()

        sentences = nltk.sent_tokenize(page_text)
        alt_corpus=preprocess_alt_text(sentences)
        altt_corpus+=alt_corpus
        preprocessed_page_text = []
        for sentence in sentences:
            preprocessed_sentence = preprocess_text(sentence)
#             preprocessed_page_text.append(preprocessed_sentence)
            preprocessed_page_text.append((sentence, preprocessed_sentence))
        preprocessed_text_corpus.append(preprocessed_page_text)
#         print(page_text)
    pdf_document.close()
    return preprocessed_sentence,preprocessed_text_corpus,altt_corpus

###Creating Corpus

In [10]:
pdf_path = r"Downloads/NLP.pdf"

preprocessed_sentence,preprocessed_text_corpus,altt_corpus = process_pdf(pdf_path)
corpus=[]
corpuss=[]

for page_corpus in preprocessed_text_corpus:
    for sentence, tokens in page_corpus:
        corpuss.append(( tokens, sentence)) 
        corpus.append(tokens)

### Bag of words

- The representation generated is too large and can't take care of words that are not in vocubulary and also the context

In [11]:
from sklearn.feature_extraction.text import CountVectorizer
count_vect = CountVectorizer() 
BOW = count_vect.fit_transform(altt_corpus) 
print("Size of vocabulary: ", len(count_vect.vocabulary_))
print("Vocabulary:",count_vect.vocabulary_)

Size of vocabulary:  263
Vocabulary: {'india': 113, 'political': 174, 'structure': 225, 'world': 261, 'large': 131, 'democracy': 57, 'multitiered': 150, 'system': 231, 'governance': 96, 'operate': 164, 'parliamentary': 170, 'president': 183, 'head': 103, 'state': 221, 'prime': 184, 'minister': 146, 'government': 97, 'country': 48, 'follow': 90, 'federal': 82, 'mean': 143, 'power': 181, 'share': 212, 'central': 24, 'union': 249, 'territory': 242, 'unify': 248, 'constitution': 41, 'indian': 114, 'parliament': 169, 'bicameral': 18, 'consist': 40, 'lok': 138, 'sabha': 204, 'house': 109, 'people': 171, 'rajya': 189, 'council': 47, 'states': 222, 'elections': 72, 'hold': 107, 'regularly': 196, 'national': 155, 'level': 134, 'witness': 260, 'significant': 214, 'transformation': 247, 'postindependence': 179, 'era': 77, 'nehruvian': 158, 'socialism': 215, 'economic': 67, 'liberalization': 135, 'also': 10, 'founding': 92, 'member': 144, 'various': 254, 'international': 122, 'organization': 167, 

In [12]:
bow_df = pd.DataFrame(BOW.toarray(), columns=count_vect.get_feature_names_out())
sorted_bow_df = bow_df.apply(lambda x: x.sort_values(ascending=False), axis=1)

print("Sorted BOG DataFrame:")
print(sorted_bow_df)

Sorted BOG DataFrame:
    accompany  achieve  add  additionally  address  advancements  agreement  \
0           0        0    0             0        0             0          0   
1           0        0    0             0        0             0          0   
2           0        0    0             0        0             0          0   
3           0        0    0             0        0             0          0   
4           0        0    0             0        0             0          0   
5           0        0    0             0        0             0          0   
6           0        0    0             0        0             0          0   
7           0        0    0             0        0             0          0   
8           0        0    0             0        0             0          0   
9           0        0    0             0        0             0          0   
10          0        0    0             0        0             0          0   
11          1        0    0   

In [13]:
from sklearn.metrics.pairwise import cosine_similarity
word1_index = count_vect.vocabulary_['india']  
word2_index = count_vect.vocabulary_['indian']  

word1_vector = BOW[:, word1_index]
word2_vector = BOW[:, word2_index]

word1_vector = word1_vector.reshape(1, -1)
word2_vector = word2_vector.reshape(1, -1)

cosine_similarity_word1_word2 = cosine_similarity(word1_vector, word2_vector)

print(f"Similarity between 'word1' and 'word2': {cosine_similarity_word1_word2[0][0]}")


Similarity between 'word1' and 'word2': 0.10050378152592122


TF-IDF

Similar to BOW, the representation generated is too large and can't take care of words that are not in vocubulary and also the context but atleast it gives an idea on how the words are related to the documents

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
tr_idf_model  = TfidfVectorizer()
tf_idf_vector = tr_idf_model.fit_transform(altt_corpus)
print("Size of vocabulary: ", len(tr_idf_model.vocabulary_))
print("Vocabulary:",tr_idf_model.vocabulary_)

Size of vocabulary:  263
Vocabulary: {'india': 113, 'political': 174, 'structure': 225, 'world': 261, 'large': 131, 'democracy': 57, 'multitiered': 150, 'system': 231, 'governance': 96, 'operate': 164, 'parliamentary': 170, 'president': 183, 'head': 103, 'state': 221, 'prime': 184, 'minister': 146, 'government': 97, 'country': 48, 'follow': 90, 'federal': 82, 'mean': 143, 'power': 181, 'share': 212, 'central': 24, 'union': 249, 'territory': 242, 'unify': 248, 'constitution': 41, 'indian': 114, 'parliament': 169, 'bicameral': 18, 'consist': 40, 'lok': 138, 'sabha': 204, 'house': 109, 'people': 171, 'rajya': 189, 'council': 47, 'states': 222, 'elections': 72, 'hold': 107, 'regularly': 196, 'national': 155, 'level': 134, 'witness': 260, 'significant': 214, 'transformation': 247, 'postindependence': 179, 'era': 77, 'nehruvian': 158, 'socialism': 215, 'economic': 67, 'liberalization': 135, 'also': 10, 'founding': 92, 'member': 144, 'various': 254, 'international': 122, 'organization': 167, 

In [15]:
tfidf_df = pd.DataFrame(tf_idf_vector.toarray(), columns=tr_idf_model.get_feature_names_out())

sorted_tfidf_df = tfidf_df.apply(lambda x: x.sort_values(ascending=False), axis=1)

print("Sorted TF-IDF DataFrame:")
print(sorted_tfidf_df)

Sorted TF-IDF DataFrame:
    accompany   achieve       add  additionally   address  advancements  \
0    0.000000  0.000000  0.000000      0.000000  0.000000      0.000000   
1    0.000000  0.000000  0.000000      0.000000  0.000000      0.000000   
2    0.000000  0.000000  0.000000      0.000000  0.000000      0.000000   
3    0.000000  0.000000  0.000000      0.000000  0.000000      0.000000   
4    0.000000  0.000000  0.000000      0.000000  0.000000      0.000000   
5    0.000000  0.000000  0.000000      0.000000  0.000000      0.000000   
6    0.000000  0.000000  0.000000      0.000000  0.000000      0.000000   
7    0.000000  0.000000  0.000000      0.000000  0.000000      0.000000   
8    0.000000  0.000000  0.000000      0.000000  0.000000      0.000000   
9    0.000000  0.000000  0.000000      0.000000  0.000000      0.000000   
10   0.000000  0.000000  0.000000      0.000000  0.000000      0.000000   
11   0.218984  0.000000  0.000000      0.000000  0.000000      0.000000   


In [16]:
word1_index = count_vect.vocabulary_['india'] 
word2_index = count_vect.vocabulary_['cuisine']  

word1_vector = tf_idf_vector[:, word1_index]
word2_vector = tf_idf_vector[:, word2_index]

word1_vector = word1_vector.reshape(1, -1)
word2_vector = word2_vector.reshape(1, -1)

cosine_similarity_word1_word2 = cosine_similarity(word1_vector, word2_vector)

print(f"Similarity between 'word1' and 'word2': {cosine_similarity_word1_word2[0][0]}")


Similarity between 'word1' and 'word2': 0.14428945260769818


In [17]:
word1_index = count_vect.vocabulary_['infosys']
word1_vector = tf_idf_vector[:, word1_index]
print(f"Word1 Vector: {word1_vector}")


Word1 Vector:   (15, 0)	0.2541654767227924


CBOW

Better representation since it can take context into account

In [18]:
from gensim.models import Word2Vec
model_cbow = Word2Vec(corpus, min_count=1, vector_size=60, window=2, sg=0)
model_cbow.train(corpus, total_examples=len(corpus), epochs=250)

(73297, 100000)

In [19]:
len(corpus)

29

In [20]:
word_vectors_cbow = model_cbow.wv

In [21]:
for word in word_vectors_cbow.index_to_key:
    print(f"Word: {word}")
    print(f"Vector: {word_vectors_cbow[word]}")
    print()

Word: india
Vector: [ 0.23542544  0.21085693 -0.09781962  0.06517088 -0.1157745  -0.00806716
  0.35961652  0.320306    0.04499314  0.03682575  0.37312454  0.13151774
  0.0649586  -0.23803999  0.0216037  -0.18282814  0.24483788  0.00230295
 -0.48137975 -0.17106846 -0.08012373 -0.08562917  0.07824343  0.16249186
  0.14140785 -0.15222411 -0.02063736  0.21986324 -0.18349497 -0.1302701
  0.10284295 -0.50163585  0.16019654 -0.03001712  0.26663646  0.2337936
  0.07958961 -0.29039392 -0.39946306  0.01296668 -0.00967625  0.1794984
 -0.29962224  0.10660733 -0.05180064 -0.10664998 -0.0401576   0.19626033
 -0.11480661  0.29753715 -0.08185206  0.12957454  0.0828144   0.30653116
  0.10441181  0.27734014  0.0043825  -0.00570484  0.4449969  -0.23363149]

Word: 's
Vector: [ 0.21744801  0.11544054 -0.16201879 -0.03652724 -0.04410494  0.0393395
  0.32118595  0.2855125   0.04223598  0.05518749  0.35148722  0.14772637
 -0.01894124 -0.19956781  0.06609529 -0.15143964  0.21562259 -0.02563526
 -0.42462873 -0.

In [24]:
word_vectors_cbow = model_cbow.wv
similarity = word_vectors_cbow.similarity('indian', 'cuisine')
print(f"Similarity between 'indian' and 'cuisine': {similarity}")

Similarity between 'indian' and 'cuisine': 0.9814869165420532


In [25]:
model_skip = Word2Vec(corpus, min_count=1, vector_size=60, window=2, sg=1)
model_skip.train(corpus, total_examples=len(corpus), epochs=200)

(58617, 80000)

In [26]:
word_vectors_skip = model_skip.wv

Skipgram

In [27]:
for word in word_vectors_skip.index_to_key:
    print(f"Word: {word}")
    print(f"Vector: {word_vectors_skip[word]}")
    print()

Word: india
Vector: [ 0.20731492  0.23001698  0.07631577  0.21664648 -0.20397119  0.08665024
  0.08135051  0.29687235  0.0627909  -0.03295257  0.27104998  0.08684203
  0.20317708 -0.33246347 -0.14287178 -0.2546762   0.22864866  0.05228009
 -0.38261464 -0.04988883  0.04095191 -0.01702704 -0.02908878  0.23147403
  0.18982933 -0.08025507 -0.03683212  0.15347748 -0.20301321 -0.24984601
 -0.0040744  -0.4419613   0.00661269 -0.07951995  0.2214339   0.15564452
  0.06351949 -0.07418281 -0.35211194  0.01223585 -0.02413006  0.22831361
 -0.223125    0.10153306 -0.1364624  -0.04991538 -0.07628082  0.22449969
 -0.07788621  0.14676355 -0.06808329  0.07901911  0.01792613  0.21446477
  0.05969974  0.19145375  0.09897981 -0.05930461  0.29261443 -0.15305431]

Word: 's
Vector: [ 0.234871   -0.03807865 -0.09936091 -0.04465369 -0.05907367  0.18800966
  0.14170355  0.2845469   0.07883595  0.00171888  0.3822201   0.1622209
  0.02263534 -0.23483358 -0.07482645 -0.20310734  0.15740861  0.00144622
 -0.32740644 

In [28]:
similarity = word_vectors_skip.similarity('indian', 'cuisine')
print(f"Similarity between word1 and word2: {similarity}")

Similarity between word1 and word2: 0.899993896484375


GloVe

In [29]:
model_glo = Word2Vec(sentences=corpus, min_count=1, vector_size=60, window=2, sg=0)
model_glo.train(corpus, total_examples=len(corpus), epochs=250)

word_vectors = model_glo.wv
for word in word_vectors.index_to_key:
    print(f"Word: {word}")
    print(f"Vector: {word_vectors[word]}")
    print()

Word: india
Vector: [ 0.23542544  0.21085693 -0.09781962  0.06517088 -0.1157745  -0.00806716
  0.35961652  0.320306    0.04499314  0.03682575  0.37312454  0.13151774
  0.0649586  -0.23803999  0.0216037  -0.18282814  0.24483788  0.00230295
 -0.48137975 -0.17106846 -0.08012373 -0.08562917  0.07824343  0.16249186
  0.14140785 -0.15222411 -0.02063736  0.21986324 -0.18349497 -0.1302701
  0.10284295 -0.50163585  0.16019654 -0.03001712  0.26663646  0.2337936
  0.07958961 -0.29039392 -0.39946306  0.01296668 -0.00967625  0.1794984
 -0.29962224  0.10660733 -0.05180064 -0.10664998 -0.0401576   0.19626033
 -0.11480661  0.29753715 -0.08185206  0.12957454  0.0828144   0.30653116
  0.10441181  0.27734014  0.0043825  -0.00570484  0.4449969  -0.23363149]

Word: 's
Vector: [ 0.21744801  0.11544054 -0.16201879 -0.03652724 -0.04410494  0.0393395
  0.32118595  0.2855125   0.04223598  0.05518749  0.35148722  0.14772637
 -0.01894124 -0.19956781  0.06609529 -0.15143964  0.21562259 -0.02563526
 -0.42462873 -0.

In [30]:
similarity = word_vectors.similarity('indian', 'cuisine')
print(f"Similarity between word1 and word2: {similarity}")

Similarity between word1 and word2: 0.9814869165420532


Fast Text

In [31]:
from gensim.models import FastText

model_fast = FastText(sentences=corpus, vector_size=60, window=5, min_count=1, sg=1)
model_fast.train(corpus, total_examples=len(corpus), epochs=250)

word_vectors = model_fast.wv
for word in word_vectors.index_to_key:
    print(f"Word: {word}")
    print(f"Vector: {word_vectors[word]}")

Word: india
Vector: [-0.12429433 -0.16308     0.33608994 -0.02805943 -0.55333465  0.20935099
 -0.3439961   0.02414824  0.25021836  0.47325227 -0.14114115 -0.14647385
  0.32523763 -0.1834807  -0.43733332 -0.2736379  -0.635216   -0.15086865
 -0.19190764  0.3322042   0.40826812  0.11954947 -0.08193512  0.37939367
  0.5878777  -0.30004025 -0.3738107   0.3969151  -0.22235674  0.30227223
 -0.03856375  0.25009635  0.47016144 -0.34576523  0.29266706  0.09235869
 -0.24563567 -0.3810724  -0.31728548 -0.15925407  0.14074771  0.27869374
 -0.01387835 -0.10129403 -0.10601569 -0.28201967  0.0788634   0.16575684
 -0.33156258 -0.15692003  0.2828733   0.06493739  0.09902924  0.32990873
  0.18575732 -0.17704864 -0.07746352  0.6010769   0.25646147 -0.07653081]
Word: 's
Vector: [-0.16961414 -0.00691388  0.38815427 -0.5285958  -0.3685006   0.7062972
 -0.10838965  0.16382605 -0.14507887 -0.0390401   0.2438514   0.42111814
 -0.23104717 -0.23326159  0.11185496 -0.42591536 -0.4954923  -0.19368424
  0.06621312 -

In [32]:
pip install sentence-transformers


DEPRECATION: Loading egg at /Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/jupyter-1.0.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
Note: you may need to restart the kernel to use updated packages.


In [33]:
similarity = word_vectors.similarity('indian', 'cuisine')
print(f"Similarity between 'indian' and 'cuisine': {similarity}")

Similarity between 'indian' and 'cuisine': 0.6447808742523193


In [34]:
def get_sentence_embedding(sentence_tokens, model):
    word_embeddings = []
    for token in sentence_tokens:
        if token in model.wv.key_to_index:  
            word_embeddings.append(model.wv[token])
    
    if len(word_embeddings) == 0:
        return None
    
    sentence_embedding = sum(word_embeddings) / len(word_embeddings)
    return sentence_embedding

In [35]:
sentence_embedding = get_sentence_embedding(preprocessed_sentence, model_skip)
print("Sentence Embedding:", sentence_embedding)

Sentence Embedding: [ 0.19696768  0.04455308 -0.03630652  0.08714023  0.11487687  0.03837568
  0.3794873   0.3023455   0.15253319 -0.04802395  0.33057076  0.10808925
 -0.10825074 -0.07835227  0.25592226 -0.15252706  0.20437957 -0.04036487
 -0.24201407 -0.22519042  0.02131158 -0.15429537  0.04473823  0.08798445
  0.14990094 -0.09353474  0.04802702  0.08740417 -0.08581828 -0.12311506
  0.0363552  -0.28705752  0.1796845  -0.07442833  0.14541417  0.18428728
  0.1343963  -0.39464945 -0.34732363 -0.04013183 -0.02773791 -0.06021317
 -0.28712758  0.18365356  0.17736895 -0.04970984  0.15568878  0.12247944
 -0.21275963  0.2916683  -0.24668053  0.1546607   0.12774293  0.29902023
  0.11837979  0.37951347 -0.07282668  0.04485183  0.3364636  -0.24523354]


In [36]:
def cosine_similarity(vector1, vector2):
    dot_product = np.dot(vector1, vector2)
    norm_vector1 = np.linalg.norm(vector1)
    norm_vector2 = np.linalg.norm(vector2)
    similarity = dot_product / (norm_vector1 * norm_vector2)
    return similarity

In [37]:
from sklearn.metrics.pairwise import cosine_similarity

def find_most_relevant_sentences_using_word_embeddings(question, corpus, model, top_n=3):
    preprocessed_question = preprocess_text(question)
    question_embedding = np.zeros((1, model.vector_size))  # Initialize question embedding with zeros
    count = 0  # Initialize count to keep track of valid tokens

    # Compute average question embedding
    for token in preprocessed_question:
        if token in model.wv.key_to_index:
            question_embedding += model.wv[token]
            count += 1

    if count == 0:
        return "Unable to find relevant sentences."

    question_embedding /= count  # Average the question embedding

    top_sentences = []
    for sentence_tokens, original_sentence in corpus:
        sentence_embedding = np.zeros((1, model.vector_size))  # Initialize sentence embedding with zeros
        count = 0  # Reset count for each sentence

        # Compute average sentence embedding
        for token in sentence_tokens:
            if token in model.wv.key_to_index:
                sentence_embedding += model.wv[token]
                count += 1

        if count > 0:
            sentence_embedding /= count  # Average the sentence embedding
            similarity = cosine_similarity(question_embedding, sentence_embedding)
            top_sentences.append((original_sentence, similarity))

    top_sentences.sort(key=lambda x: x[1], reverse=True)
    top_sentences = top_sentences[:top_n]

    return top_sentences


In [38]:
pip install sentence-transformers


DEPRECATION: Loading egg at /Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/jupyter-1.0.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
Note: you may need to restart the kernel to use updated packages.


In [39]:
from sentence_transformers import SentenceTransformer

# Example usage to ensure the module is working
model = SentenceTransformer('all-MiniLM-L6-v2')
sentences = ['This is an example sentence.', 'Each sentence is transformed into a vector.']
embeddings = model.encode(sentences)

print(embeddings)

[[ 9.81245339e-02  6.78126588e-02  6.25231862e-02  9.50848311e-02
   3.66477184e-02 -3.98467528e-03  7.47757172e-03 -1.32314852e-02
   6.28837049e-02  2.24954803e-02  7.26957545e-02 -3.12742405e-02
   4.63550836e-02 -1.25545664e-02  4.78147268e-02 -4.91032330e-03
   4.94200401e-02 -6.41091987e-02 -9.69658270e-02  3.28888185e-02
   5.41044101e-02  3.53285708e-02  3.30505073e-02  1.46993119e-02
  -3.34306285e-02 -2.56156921e-02 -5.07921837e-02  7.32544959e-02
   1.10273942e-01 -2.96618231e-02 -6.75570741e-02 -3.05714067e-02
   3.95602770e-02  4.54760119e-02  1.59961153e-02  3.85504030e-02
  -1.09541081e-02  8.48356709e-02 -4.42870595e-02 -6.79648388e-03
   9.42569971e-03  5.08141675e-05  1.30363111e-03 -1.19697908e-02
   1.36451572e-02 -8.41742530e-02 -1.65159610e-04  5.48379775e-03
   2.56150868e-02 -3.15452814e-02 -1.07344687e-01 -4.57878597e-02
  -9.11749974e-02 -2.51047360e-03  1.79983806e-02  4.94016111e-02
   6.18487922e-03  5.97963892e-02  2.70025749e-02 -1.61221977e-02
  -1.81497

In [40]:
from sentence_transformers import SentenceTransformer, util

model_sbert = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def calculate_similarity_st(sent_embedding1, sent_embedding2):
    return util.pytorch_cos_sim(sent_embedding1, sent_embedding2)

def find_most_relevant_sentences_using_sentence_transformers(question, corpus, model_sbert, top_n=3):
    question_embedding = model_sbert.encode([question], convert_to_tensor=True)

    top_sentences = []
    for sentence_tokens, original_sentence in corpus:
        sentence = ' '.join(sentence_tokens)
        sentence_embedding = model_sbert.encode([sentence], convert_to_tensor=True)
        similarity = calculate_similarity_st(question_embedding, sentence_embedding)
        top_sentences.append((original_sentence, similarity.item()))

    top_sentences.sort(key=lambda x: x[1], reverse=True)
    top_sentences = top_sentences[:top_n]

    return top_sentences



In [41]:
from sklearn.metrics.pairwise import cosine_similarity
def find_most_relevant_sentences_using_tfidf(question_vector, corpus_vectors, corpus_sentences, top_n=3):
    top_sentences = []
    for sentence_vector, original_sentence in zip(corpus_vectors, corpus_sentences):
        similarity = cosine_similarity(question_vector, sentence_vector)
        top_sentences.append((original_sentence, similarity))

    top_sentences.sort(key=lambda x: x[1], reverse=True)
    top_sentences = top_sentences[:top_n]

    return top_sentences


In [42]:
from transformers import AutoTokenizer, AutoModel
import torch

BERT_Model = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(BERT_Model)
model_bert = AutoModel.from_pretrained(BERT_Model)

def sent_embedding(sent):
    tokens = tokenizer.encode_plus(sent, max_length=128, truncation=True,
                                    padding='max_length', return_tensors='pt')
    with torch.no_grad():
        outputs = model_bert(**tokens)
        embedding = outputs.pooler_output.detach().numpy()
    return embedding

def calculate_similarity(sent_embedding1, sent_embedding2):
    sent_embedding1 = torch.tensor(sent_embedding1)
    sent_embedding2 = torch.tensor(sent_embedding2)
    return torch.nn.functional.cosine_similarity(sent_embedding1, sent_embedding2).item()

def find_most_relevant_sentences_using_bert(question, corpus, model, top_n=3):
    question_embedding = sent_embedding(question)

    top_sentences = []
    for sentence_tokens, original_sentence in corpus:
        sentence = ' '.join(sentence_tokens)
        sentence_embedding = sent_embedding(sentence)
        similarity = calculate_similarity(question_embedding, sentence_embedding)
        top_sentences.append((original_sentence, similarity))

    top_sentences.sort(key=lambda x: x[1], reverse=True)
    top_sentences = top_sentences[:top_n]

    return top_sentences


In [48]:
user_question = input("Enter your question: ")
a=input('Choose Model:\n1)Bag of Words\n2)Tf-Idf\n3)CBOW\n4)Skip Gram\n5)Glove\n6)FastText\n7)SentenceTransformer\n8)BERT\n')
if a=='1':
    print('Bag of Words')
    question_vector = count_vect.transform([user_question])
    top_relevant_sentences = find_most_relevant_sentences_using_tfidf(question_vector, BOW, altt_corpus, top_n=3)
elif a=='2':
    print('Tf-Idf')
    question_vector = tr_idf_model.transform([user_question])
    top_relevant_sentences = find_most_relevant_sentences_using_tfidf(question_vector, tf_idf_vector, altt_corpus, top_n=3)
elif a=='3':
    print('CBOW')
    top_relevant_sentences = find_most_relevant_sentences_using_word_embeddings(user_question, corpuss, model_cbow, top_n=3)
elif a=='4':
    print('Skip Gram')
    top_relevant_sentences = find_most_relevant_sentences_using_word_embeddings(user_question, corpuss, model_skip, top_n=3)
elif a=='5':
    print('Glove')
    top_relevant_sentences = find_most_relevant_sentences_using_word_embeddings(user_question, corpuss, model_glo, top_n=3)
elif a=='6':
    print('Fast Text')
    top_relevant_sentences = find_most_relevant_sentences_using_word_embeddings(user_question, corpuss, model_fast, top_n=3)
elif a=='7':
    print('Sentence Transformer')
    top_relevant_sentences = find_most_relevant_sentences_using_sentence_transformers(user_question, corpuss, model_sbert, top_n=3)
elif a=='8':
    print('BERT')
    top_relevant_sentences = find_most_relevant_sentences_using_bert(user_question, corpuss, model_bert, top_n=3)
else:
    print("Invalid Choice")

def capitalize_first_letter(sentence):
    if sentence:
        return sentence[0].upper() + sentence[1:]
    return ""    
    
required_words = ["In addition", "Moreover"]
required_punctuation = "."
modified_content = ""

if not top_relevant_sentences:
    modified_content = "Unable to find relevant sentences."
elif top_relevant_sentences== "Unable to find relevant sentences.":
    modified_content = "Unable to find relevant sentences."
else:
    for i, (sentence, _) in enumerate(top_relevant_sentences):
        sentence = sentence.replace('\n', '')
        modified_sentence = capitalize_first_letter(' '.join(sentence.split()))
        if(modified_sentence[-1]=='.'):
            modified_content += modified_sentence 
        else:
            modified_content += modified_sentence+required_punctuation
        if i < len(required_words) and len(top_relevant_sentences) > i:  
            modified_content += ' ' + required_words[i]+' '

print("\nModified Content:")
print(modified_content)


Enter your question:  Who is the head of state and who is the head of government in India?
Choose Model:
1)Bag of Words
2)Tf-Idf
3)CBOW
4)Skip Gram
5)Glove
6)FastText
7)SentenceTransformer
8)BERT
 1


Bag of Words

Modified Content:
Operate parliamentary system president head state prime minister head government. In addition State union territory government unify constitution india. Moreover Country follow federal structure mean power share central government state.
